# Perch 2 Temporal Layer — Pipeline (local, spark-ae0e)

Cleans up per-clip false detections (**humpback / orca / PWSD**) by smoothing the
sequence of per-frame scores with a per-species **stickiness HMM**, then (later) a
learned BiLSTM/TCN over the same frame stack.

**Runtime.** The heavy inference is already done: the Google *multispecies* model
was run over months of audio and lives in your JSONs, and **Perch runs on your
Spark box**. So this notebook is light — **a CPU high-RAM runtime or a T4 is
plenty; an A100 is unnecessary.** Use a T4 only if you enable the optional
in-notebook humpback confirmatory model (Cell 6).

**Drive layout — everything under `MyDrive/perch2_temporal`:**

```
perch2_temporal/
  input/
    multispecies_json/   raw per-5s-chunk JSONs from the model (source of truth)
    multispecies_csv/    consolidated per-recording wide logit CSVs (pipeline reads these)
    perch/               Perch exports from Spark (.npz per recording)
    labels/              calibration labels (labels.csv)
    audio/               OPTIONAL raw 32 kHz wav, only for Cell 5
  output/                intervals + smoothed posteriors written here
```

**Ingest note.** The model emits ~120 tiny JSONs per 10-min recording
(~500k/month) — too many small files to read off Drive at scale. Cell 3
consolidates each recording's chunks into ONE wide CSV of **logits**
(`epoch_seconds, Oo_logit, Mn_logit, ...`), reading the class-aligned
`all_logits` — never the JSON's `scores` field, which is sorted and unaligned.
The pipeline then reads those compact CSVs.

Cells: **1** mount · **2** config · **3** consolidate JSON → wide CSV + load ·
**4** Perch handoff · **5** optional humpback model · **6** frame stack ·
**7** calibration · **8** HMM · **9** stitch+intervals · **10** run+save.

In [ ]:
# === 1. Paths (local, on spark-ae0e) =======================================
# Runs directly on spark-ae0e against the mounted archive -- no Drive, no Colab.
# The loaders glob recursively, so pointing at the logits/ and perch/ roots picks
# up 2018/04 (and any future month) with no code change.
import os
GMWD   = '/mnt/PAM_Analysis/GoogleMultiSpeciesWhaleModel2'
PATHS = {
    'ms_csv':  f'{GMWD}/logits',                       # consolidated *_logits.csv tree
    'perch':   f'{GMWD}/perch',                         # slim per-recording *.npz tree
    'labels':  '/mnt/PAM_Analysis/perch-hoplite/labels',
    'audio':   f'{GMWD}/resampled_32kHz',               # only for the optional humpback model
    'output':  os.path.expanduser('~/perch-temporal/output'),
}
os.makedirs(PATHS['output'], exist_ok=True)             # only create what we write
print('inputs:', GMWD)
for k in ('ms_csv', 'perch'):
    print(f'  {k}: {PATHS[k]}  exists={os.path.isdir(PATHS[k])}')

In [ ]:
# === 2. Config & helpers ===================================================
import re, json, glob, datetime as dt
from dataclasses import dataclass, field
from typing import Optional
import numpy as np

SPECIES = ("humpback", "orca", "pwsd")

# Canonical multispecies class order = the model card's index map, VERIFIED here
# against the internally-aligned class_names/scores pair across chunks from
# multiple recordings. This is the true order of all_logits / all_probabilities.
# (It is NOT the per-row-sorted class_names field, which changes every chunk.)
MS_CLASS_ORDER = ["Oo", "Mn", "Eg", "Be", "Upcall", "Bp",
                  "Call", "Gunshot", "Echolocation", "Bm", "Whistle", "Ba"]

@dataclass
class Config:
    # Your precomputed multispecies chunks are 5 s non-overlapping, so the common
    # grid is 5 s / 5 s. Run Perch on Spark at the SAME 5 s hop to align.
    frame_sec: float = 5.0
    hop_sec:   float = 5.0
    use_humpback_model: bool = False        # Cell 5; needs T4 + raw audio in Drive

    # Raw channels feeding each species HMM (before calibration). Perch channels
    # are the orca_v4 logits; ms_* are the multispecies logits. perch_other and
    # perch_ship_noise are NOT species -- they ride along as features/covariates
    # (see build_frame_stacks), useful as background/noise context for the HMM.
    emission_sources: dict = field(default_factory=lambda: {
        "humpback": ["perch_humpback_song", "ms_Mn"],
        "orca":     ["perch_orca_call",     "ms_Oo"],
        "pwsd":     ["perch_dolphin_call"],  # PWSD == dolphin_call; label desert, Perch only
    })
    # Duration priors -> stickiness. expected_bout_frames = 1/(1-p_self).
    # PLACEHOLDERS — set from your annotations / field knowledge.
    expected_present_sec: dict = field(default_factory=lambda: {
        "humpback": 600.0, "orca": 180.0, "pwsd": 90.0})
    expected_absent_sec: dict = field(default_factory=lambda: {
        "humpback": 1800.0, "orca": 1800.0, "pwsd": 1800.0})
    class_prior: dict = field(default_factory=lambda: {
        "humpback": None, "orca": None, "pwsd": None})   # scaled-likelihood opt-in
    outage_gap_sec: float = 120.0            # gaps larger than this split the timeline

cfg = Config()

_TS = re.compile(r"(\d{8})[_T](\d{6})")          # 20180401T000914 or 20180401_000914
def ts_key(name: str) -> str:
    m = _TS.search(name)
    if not m:
        raise ValueError(f"no YYYYMMDD[_/T]HHMMSS timestamp in {name!r}")
    return m.group(1) + "T" + m.group(2)
def ts_epoch(key: str) -> float:                 # UTC, to match the epoch_seconds column
    return dt.datetime.strptime(key, "%Y%m%dT%H%M%S").replace(
        tzinfo=dt.timezone.utc).timestamp()

In [ ]:
# === 3. Consolidate per-chunk JSON -> wide logit CSV, then load ============
# Your JSON: one per 5 s chunk, keys: class_names, all_logits, all_probabilities,
# scores. USE all_logits (class-aligned). The "scores" field is sorted descending
# and NOT aligned to class_names -- do not use it.
CHUNK_RE = re.compile(r"chunk_(\d+)", re.I)

def _order_ok(class_names, scores, all_probabilities, tol=1e-9):
    """all_logits/all_probabilities are in MS_CLASS_ORDER. Verify against the
    internally-aligned (class_names, scores) pair; returns False on any mismatch."""
    truth = dict(zip(class_names, scores))                     # class -> prob
    try:
        want = [truth[c] for c in MS_CLASS_ORDER]
    except KeyError:
        return False
    return all(abs(a - b) <= tol for a, b in zip(want, all_probabilities))

def consolidate_recording(json_dir, out_csv, hop_sec=5.0):
    """All chunk-JSONs for ONE recording -> one wide CSV in canonical order:
       epoch_seconds, Oo_logit, Mn_logit, ...  (all_logits is in MS_CLASS_ORDER;
       we assert that per file via the class_names/scores pair)."""
    files = glob.glob(f"{json_dir}/**/*chunk_*.json", recursive=True)
    files = sorted(files, key=lambda p: int(CHUNK_RE.search(p).group(1)))
    if not files:
        raise FileNotFoundError(f"no chunk_*.json under {json_dir}")
    t0 = ts_epoch(ts_key(os.path.basename(files[0])))          # recording start (UTC)
    rows = []
    for fp in files:
        o = json.load(open(fp))
        if not _order_ok(o["class_names"], o["scores"], o["all_probabilities"]):
            raise ValueError(f"{fp}: all_logits order != MS_CLASS_ORDER "
                             f"-- do not trust; check the class list")
        idx = int(CHUNK_RE.search(fp).group(1))                # chunk_001 -> offset 0
        rows.append([t0 + (idx - 1) * hop_sec] + list(o["all_logits"]))
    import csv
    with open(out_csv, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["epoch_seconds"] + [f"{c}_logit" for c in MS_CLASS_ORDER])
        w.writerows(rows)
    print(f"  {os.path.basename(out_csv)}: {len(rows)} chunks x {len(MS_CLASS_ORDER)} classes")

def consolidate_all(json_root=None, csv_out=None, hop_sec=5.0):
    """FALLBACK ONLY. Normally consolidation runs on spark-ae0e via the standalone
       consolidate_multispecies.py (resumable, logs skips); the CSVs already exist
       under PATHS['ms_csv']. This in-notebook version consolidates scores_gpu ->
       one CSV per recording if you ever need it here.
       Each subfolder of json_root holding a recording's chunks -> one CSV."""
    json_root = json_root or '/mnt/PAM_Analysis/GoogleMultiSpeciesWhaleModel2/scores_gpu'
    csv_out = csv_out or PATHS['ms_csv']
    recs = sorted({os.path.dirname(p) for p in
                   glob.glob(f"{json_root}/**/*chunk_*.json", recursive=True)})
    print(f"{len(recs)} recordings to consolidate")
    for d in recs:
        out = f"{csv_out}/{os.path.basename(d)}_logits.csv"
        if not os.path.exists(out):
            consolidate_recording(d, out, hop_sec)

def _to_logit(a, eps=1e-7):
    a = np.clip(np.asarray(a, float), eps, 1 - eps); return np.log(a / (1 - a))

def load_scores_csv(path=None):
    """dir tree of wide CSVs (recurses YYYY/MM) -> {ts_key: (epoch_seconds[T],
       {ms_<Class>: logits[T]})}. Reads *_logits.csv only, so per-class files like
       ..._epoch_oo_scores.csv sitting in the same tree are ignored. Columns
       *_logit are used as-is; *_class_score/_prob/_score are converted to logit."""
    import pandas as pd
    path = path or PATHS['ms_csv']; out = {}
    for fp in sorted(glob.glob(f"{path}/**/*_logits.csv", recursive=True)):
        df = pd.read_csv(fp).sort_values("epoch_seconds")
        epochs = df["epoch_seconds"].to_numpy(float)
        tracks = {}
        for c in df.columns:
            if c == "epoch_seconds":
                continue
            for suf, conv in (("_logit", lambda x: x),
                              ("_class_score", _to_logit), ("_prob", _to_logit),
                              ("_score", _to_logit)):
                if c.endswith(suf):
                    tracks[f"ms_{c[:-len(suf)]}"] = conv(df[c].to_numpy(float)); break
        out[ts_key(os.path.basename(fp))] = (epochs, tracks)
    print(f"loaded {len(out)} recordings from {path}")
    return out

def load_expanded_csv(path=None):
    """Read your existing *_expanded.csv directly (no JSON re-read).
       Uses all_logits_1..12 POSITIONALLY with MS_CLASS_ORDER; IGNORES the
       class_names_*/scores_* columns (they are re-sorted per row -> would
       mislabel every column after the first). Epoch reconstructed from the
       recording timestamp + 5_sec_time_offset * 5 s.
       -> {ts_key: (epoch_seconds[T], {ms_<Class>: logits[T]})}."""
    import pandas as pd
    path = path or PATHS['ms_csv']; out = {}
    for fp in sorted(glob.glob(f"{path}/**/*expanded.csv", recursive=True)):
        df = pd.read_csv(fp).sort_values("5_sec_time_offset")
        # self-check on the first row: the per-row class_names_*/scores_* pair must
        # imply MS_CLASS_ORDER for the all_probabilities_* / all_logits_* columns.
        r0 = df.iloc[0]
        if all(f"class_names_{i+1}" in df.columns for i in range(len(MS_CLASS_ORDER))):
            truth = {r0[f"class_names_{i+1}"]: r0[f"scores_{i+1}"]
                     for i in range(len(MS_CLASS_ORDER))}
            want = [truth.get(c) for c in MS_CLASS_ORDER]
            got = [r0[f"all_probabilities_{i+1}"] for i in range(len(MS_CLASS_ORDER))]
            if any(a is None or abs(a - b) > 1e-9 for a, b in zip(want, got)):
                raise ValueError(f"{os.path.basename(fp)}: all_logits_* order "
                                 f"!= MS_CLASS_ORDER -- check the class list")
        k = ts_key(os.path.basename(fp))
        epochs = ts_epoch(k) + df["5_sec_time_offset"].to_numpy(float) * 5.0
        tracks = {f"ms_{c}": df[f"all_logits_{i+1}"].to_numpy(float)
                  for i, c in enumerate(MS_CLASS_ORDER)}
        out[k] = (epochs, tracks)
    print(f"loaded {len(out)} recordings from expanded CSVs")
    return out

In [ ]:
# === 4. Perch handoff from Spark ===========================================
# export_slim_npz.py (in perch-hoplite) writes one .npz per recording into
# perch/YYYY/MM/, each carrying:
#   perch_dolphin_call / perch_humpback_song / perch_orca_call /
#   perch_other / perch_ship_noise          float32 [T]   RAW logits (orca_v4)
#   epoch_seconds                           float64 [T]   absolute UTC, from the DB
#   hop_sec                                 scalar == cfg.hop_sec (5.0)
#   embeddings                              float32 [T,1536]  ONLY if --with-embeddings
# The slim export omits embeddings (the HMM doesn't need them); load_perch keeps
# them optional so both slim and full exports work.
PERCH_CHANNELS = ("perch_dolphin_call", "perch_humpback_song", "perch_orca_call",
                  "perch_other", "perch_ship_noise")

def load_perch(path=None):
    """dir tree of .npz (recurses YYYY/MM) -> {ts_key: {t0, epochs, embeddings|None,
       perch:{channel: logits[T]}}}. Uses the file's own epoch_seconds as the time
       axis (do NOT reconstruct from the filename -- the DB is the source of truth)."""
    path = path or PATHS['perch']; out = {}
    for fp in sorted(glob.glob(f"{path}/**/*.npz", recursive=True)):
        k = ts_key(os.path.basename(fp)); z = np.load(fp, allow_pickle=True)
        hop = float(z["hop_sec"]) if "hop_sec" in z else cfg.hop_sec
        assert abs(hop - cfg.hop_sec) < 1e-6, f"{k}: Perch hop {hop} != {cfg.hop_sec}"
        epochs = z["epoch_seconds"].astype(float)
        out[k] = {"t0": float(epochs[0]), "epochs": epochs,
                  "embeddings": z["embeddings"] if "embeddings" in z.files else None,
                  "perch": {c: z[c] for c in PERCH_CHANNELS if c in z.files}}
    print(f"loaded {len(out)} Perch exports from {path}")
    return out

In [ ]:
# === 5. OPTIONAL: humpback confirmatory model (TF-Hub) =====================
# Needs a T4 + raw 32 kHz wav in input/audio/. No-ops if disabled.
_HB = None
def humpback_scores(k):
    global _HB
    if not cfg.use_humpback_model:
        return None
    import tensorflow as tf, tensorflow_hub as hub, soundfile as sf, librosa
    wavs = glob.glob(f"{PATHS['audio']}/**/*{k}*.wav", recursive=True)
    if not wavs:
        print(f"[hb] no audio for {k}; skip"); return None
    if _HB is None:
        _HB = hub.load("https://www.kaggle.com/models/google/"
                       "humpback-whale/TensorFlow2/humpback-whale/1")
    x, sr = sf.read(wavs[0], dtype="float32"); x = x if x.ndim == 1 else x.mean(1)
    x = librosa.resample(x, orig_sr=sr, target_sr=10000)
    s = _HB.signatures["score"](
        waveform=tf.constant(x[None, :, None]),
        context_step_samples=tf.cast(int(cfg.hop_sec*10000), tf.int64))["scores"]
    return {"hb_Mn": np.asarray(s)[0, :, 0]}

In [ ]:
# === 6. Assemble aligned frame stacks ======================================
@dataclass
class FrameStack:
    X: np.ndarray; feature_names: list; times: np.ndarray
    embeddings: Optional[np.ndarray] = None
    def col(self, name): return self.X[:, self.feature_names.index(name)]

def build_frame_stacks(ms, pe):
    stacks = {}
    for k in sorted(set(ms) & set(pe)):
        epochs, ms_tracks = ms[k]; p = pe[k]
        chans = dict(ms_tracks); chans.update(p["perch"])
        hb = humpback_scores(k)
        if hb: chans.update(hb)
        # all channels are 5 s from the same recording start -> align by index,
        # trim to shortest; use the absolute epoch column as the time axis.
        emb = p.get("embeddings")
        lens = [len(v) for v in chans.values()] + [len(epochs)]
        if emb is not None: lens.append(len(emb))
        n = min(lens)
        names = sorted(chans)
        X = np.column_stack([chans[c][:n] for c in names]).astype(np.float32)
        stacks[k] = FrameStack(X, names, epochs[:n],
                               emb[:n] if emb is not None else None)
    miss_ms, miss_pe = set(pe)-set(ms), set(ms)-set(pe)
    if miss_ms or miss_pe:
        print(f"unmatched: {len(miss_pe)} ms-only, {len(miss_ms)} perch-only")
    print(f"built {len(stacks)} aligned frame stacks")
    return stacks

In [ ]:
# === 7. Calibration (raw score -> probability-like) ========================
# Both model cards say scores are NOT probabilities; uncalibrated emissions make
# the HMM look broken. Fit on labels.csv columns: ts_key, frame_index, species, label
class Calibrator:
    def __init__(self, method="platt"): self.method=method; self._m={}
    def fit(self, raw, labels):
        for ch, y in labels.items():
            s = np.asarray(raw[ch]).reshape(-1, 1)
            if self.method == "platt":
                from sklearn.linear_model import LogisticRegression
                self._m[ch] = LogisticRegression(max_iter=1000).fit(s, y)
            else:
                from sklearn.isotonic import IsotonicRegression
                self._m[ch] = IsotonicRegression(out_of_bounds="clip").fit(s.ravel(), y)
        return self
    def apply(self, ch, raw):
        m = self._m.get(ch); raw = np.asarray(raw, float)
        if m is None: return 1/(1+np.exp(-raw))          # squash uncalibrated
        if self.method == "platt": return m.predict_proba(raw.reshape(-1,1))[:,1]
        return m.predict(raw)

def combine_emission(fs, cal, sources):
    ps = [cal.apply(s, fs.col(s)) for s in sources if s in fs.feature_names]
    return np.mean(np.column_stack(ps), axis=1)

def fit_calibrator_from_csv(stacks, method="platt"):
    import csv
    fp = f"{PATHS['labels']}/labels.csv"
    if not os.path.exists(fp):
        print("no labels.csv -> using uncalibrated squash (PWSD especially needs labels)")
        return Calibrator(method)
    raw, lab = {}, {}
    with open(fp) as fh:
        for r in csv.DictReader(fh):
            k = ts_key(r["ts_key"]) if _TS.search(r["ts_key"]) else r["ts_key"]
            if k not in stacks: continue
            i, y, sp = int(r["frame_index"]), int(r["label"]), r["species"]
            for ch in cfg.emission_sources.get(sp, []):
                if ch in stacks[k].feature_names and i < len(stacks[k].times):
                    raw.setdefault(ch, []).append(stacks[k].col(ch)[i])
                    lab.setdefault(ch, []).append(y)
    raw = {c: np.array(v) for c, v in raw.items()}
    lab = {c: np.array(v) for c, v in lab.items()}
    print("calibrating channels:", {c: len(v) for c, v in lab.items()})
    return Calibrator(method).fit(raw, lab)

In [ ]:
# === 8. Per-species 2-state stickiness HMM =================================
from scipy.special import logsumexp

def transition_matrix(exp_present_sec, exp_absent_sec, hop_sec):
    Ep = max(exp_present_sec/hop_sec, 1+1e-6); Ea = max(exp_absent_sec/hop_sec, 1+1e-6)
    a11 = 1 - 1/Ep; a00 = 1 - 1/Ea          # stay-present / stay-absent
    return np.array([[a00, 1-a00], [1-a11, a11]])

def emissions_logprob(p, prior, eps=1e-6):
    p = np.clip(p, eps, 1-eps)
    if prior is None:
        ea, ep = 1-p, p
    else:
        cp = float(np.clip(prior, eps, 1-eps)); ea, ep = (1-p)/(1-cp), p/cp
    return np.log(np.column_stack([ea, ep]))

def viterbi(log_e, log_A, log_pi):
    T = log_e.shape[0]; delta = np.full((T,2), -np.inf); psi = np.zeros((T,2), int)
    delta[0] = log_pi + log_e[0]
    for t in range(1, T):
        for j in range(2):
            tr = delta[t-1] + log_A[:, j]
            psi[t, j] = int(np.argmax(tr)); delta[t, j] = tr[psi[t, j]] + log_e[t, j]
    path = np.zeros(T, int); path[-1] = int(np.argmax(delta[-1]))
    for t in range(T-2, -1, -1): path[t] = psi[t+1, path[t+1]]
    return path

def forward_backward(log_e, log_A, log_pi):
    T = log_e.shape[0]; la = np.full((T,2), -np.inf); lb = np.zeros((T,2))
    la[0] = log_pi + log_e[0]
    for t in range(1, T):
        for j in range(2): la[t, j] = logsumexp(la[t-1] + log_A[:, j]) + log_e[t, j]
    for t in range(T-2, -1, -1):
        for i in range(2): lb[t, i] = logsumexp(log_A[i,:] + log_e[t+1] + lb[t+1])
    lg = la + lb; lg -= logsumexp(lg, axis=1, keepdims=True)
    return np.exp(lg[:, 1])

@dataclass
class SpeciesResult:
    species: str; path: np.ndarray; posterior: np.ndarray; intervals: list

def smooth_species(species, p, times):
    A = transition_matrix(cfg.expected_present_sec[species],
                          cfg.expected_absent_sec[species], cfg.hop_sec)
    log_A = np.log(A); base = float(np.clip(p.mean(), 1e-6, 1-1e-6))
    log_pi = np.log([1-base, base])
    log_e = emissions_logprob(p, cfg.class_prior[species])
    path = viterbi(log_e, log_A, log_pi)
    post = forward_backward(log_e, log_A, log_pi)
    return SpeciesResult(species, path, post, extract_intervals(path, times))

In [ ]:
# === 9. Intervals + cross-file stitching ===================================
def extract_intervals(path, times):
    out, start = [], None
    for i, s in enumerate(path):
        if s == 1 and start is None: start = times[i]
        elif s == 0 and start is not None:
            out.append((start, times[i-1] + cfg.frame_sec)); start = None
    if start is not None: out.append((start, times[-1] + cfg.frame_sec))
    return out

def _concat(ss):
    emb = np.vstack([s.embeddings for s in ss]) if ss[0].embeddings is not None else None
    return FrameStack(np.vstack([s.X for s in ss]), ss[0].feature_names,
                      np.concatenate([s.times for s in ss]), emb)

def stitch(stacks):
    """Concatenate timestamp-sorted recordings; split on outages so the HMM never
    smooths across a multi-hour gap. Decoding per contiguous segment also keeps
    the Viterbi sequence length bounded."""
    stacks = sorted(stacks, key=lambda s: s.times[0])
    segs, buf = [], [stacks[0]]
    for prev, cur in zip(stacks, stacks[1:]):
        if cur.times[0] - (prev.times[-1] + cfg.frame_sec) > cfg.outage_gap_sec:
            segs.append(_concat(buf)); buf = [cur]
        else: buf.append(cur)
    segs.append(_concat(buf)); return segs

In [ ]:
# === 10. Run, sanity-check, save ===========================================
def run(stacks, cal):
    res = {sp: [] for sp in SPECIES}
    for seg in stitch(list(stacks.values())):
        for sp in SPECIES:
            p = combine_emission(seg, cal, cfg.emission_sources[sp])
            res[sp].append(smooth_species(sp, p, seg.times))
    return res

def ab_check(stacks, cal, species="humpback", threshold=0.5):
    """The thesis in a few integers: how many isolated FPs the HMM erases."""
    seg = stitch(list(stacks.values()))[0]
    p = combine_emission(seg, cal, cfg.emission_sources[species])
    naive = (p >= threshold).astype(int)
    sm = smooth_species(species, p, seg.times).path
    print({"species": species, "naive_pos": int(naive.sum()),
           "smoothed_pos": int(sm.sum()),
           "flipped_to_absent": int(((naive==1)&(sm==0)).sum()),
           "flipped_to_present": int(((naive==0)&(sm==1)).sum())})

def save_outputs(res):
    import csv
    outdir = f"{PATHS['output']}/run_{dt.datetime.now():%Y%m%dT%H%M%S}"
    os.makedirs(outdir, exist_ok=True)
    with open(f"{outdir}/detections.csv", "w", newline="") as fh:
        w = csv.writer(fh); w.writerow(["species","onset_utc","offset_utc","dur_sec"])
        for sp, results in res.items():
            for r in results:
                for on, off in r.intervals:
                    w.writerow([sp, dt.datetime.utcfromtimestamp(on).isoformat(),
                                dt.datetime.utcfromtimestamp(off).isoformat(),
                                round(off-on, 1)])
    print("wrote", f"{outdir}/detections.csv"); return outdir

# --- driver ------------------------------------------------------------------
# April 2018 inputs already exist on the mount (logits/2018/04 + perch/2018/04),
# so we go straight to loading. (consolidate_all() is a fallback only; see Cell 3.)
ms = load_scores_csv()                       # reads logits/**/ *_logits.csv  (~4,303)
pe = load_perch()                            # reads perch/**/ *.npz          (~4,320)
stacks = build_frame_stacks(ms, pe)          # aligned on absolute UTC epoch
cal = fit_calibrator_from_csv(stacks)        # falls back to squash if no labels.csv
ab_check(stacks, cal, "humpback")            # sanity: watch flipped_to_absent
# res = run(stacks, cal)                      # full decode across all recordings
# save_outputs(res)

### Notes & next steps

- **Consolidate first (Cell 3).** `consolidate_all()` turns the ~500k per-chunk
  JSONs into a handful of per-recording wide CSVs of logits, reading the
  class-aligned `all_logits` (never `scores`). Run it once here or on Spark.
- **Already have `*_expanded.csv`?** Use `load_expanded_csv()` instead — no JSON
  re-read. It reads `all_logits_1..12` POSITIONALLY against `MS_CLASS_ORDER` and
  ignores `class_names_*`/`scores_*`, which are re-sorted per row and would
  mislabel every column after the first.
- **Why logits, not the probabilities:** the sigmoid outputs pile up against 0
  (e.g. Mn≈2.5e-4), so calibration is far better conditioned on the logits
  (≈ −6 to −12). `load_scores_csv` auto-converts any probability columns to logits.
- **Perch handoff (Cell 4):** `export_slim_npz.py` (in perch-hoplite) writes one
  `.npz` per recording at **5 s hop** with the five `perch_*` logit tracks +
  `epoch_seconds` (+ optional `embeddings`); `load_perch` reads them straight off
  the mount and aligns to the CSVs by absolute UTC epoch.
- **Calibrate before trusting anything** (Cell 7). PWSD is the starved channel —
  prioritise labelling there; humpback/orca can lean on the whale-model heads.
- **Tune the duration priors** in `Config` (Cell 2) from your annotations.
- **Order of operations:** get `ab_check` on humpback looking sensible, then orca,
  then PWSD. `flipped_to_absent` is the payoff metric.
- **Later:** the learned BiLSTM/TCN trains on `FrameStack.embeddings` (already
  carried through) and replaces the fixed transition prior with a learned one.
- **Speed:** Viterbi/forward-backward loop over frames but decode **per contiguous
  segment**, so length is bounded by your longest gapless run. If a run is huge,
  either lower `outage_gap_sec` or swap in `hmmlearn`.